# Water Quality Classification
Predicting water quality levels (VERDE / AMARILLO / ROJO) from chemical measurements using a Decision Tree classifier.

**Dataset:** Mexican groundwater monitoring data (2012–2024)  
**Target:** `SEMÁFORO` — a traffic-light quality indicator with three classes:
- 🟢 `VERDE` (0) — Good quality
- 🟡 `AMARILLO` (1) — Moderate concern
- 🔴 `ROJO` (2) — Poor quality

## Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

## Load Data

In [ ]:
df = pd.read_excel(Path("data") / "Calidad_del_Agua_Subterr_nea_p_2012-2024_15082025.xlsx")

In [ ]:
df.head()

In [ ]:
df.columns

In [ ]:
df["SEMÁFORO"].unique()

## Feature Selection

We keep only the 14 chemical measurement columns and the target variable `SEMAFORO`,
dropping metadata columns like site name, location, and administrative region.

In [ ]:
# Measured Chemical Values
features = ['ALC_mg/L',
            'CONDUCT_mS/cm',
            'SDT_mg/L',
            'FLUORUROS_mg/L',
            'DUR_mg/L',
            'COLI_FEC_NMP/100_mL',
            'N_NO3_mg/L',
            'AS_TOT_mg/L',
            'CD_TOT_mg/L',
            'CR_TOT_mg/L',
            'HG_TOT_mg/L',
            'PB_TOT_mg/L',
            'MN_TOT_mg/L',
            'FE_TOT_mg/L',
            'SEMÁFORO'
]

df = df[features]

## Data Cleaning

Most chemical columns were stored as `object` instead of `float` because some values
were recorded as detection limits (e.g. `<0.001`). We strip the `<` prefix and treat
those values as `0`, then convert all columns to `float`.

In [ ]:
df.info()

In [ ]:
# Columns to convert (all except SEMAFORO and CONDUCT which is already float)
cols_to_convert = [
    'ALC_mg/L', 'SDT_mg/L', 'FLUORUROS_mg/L', 'DUR_mg/L',
    'COLI_FEC_NMP/100_mL', 'N_NO3_mg/L', 'AS_TOT_mg/L',
    'CD_TOT_mg/L', 'CR_TOT_mg/L', 'HG_TOT_mg/L',
    'PB_TOT_mg/L', 'MN_TOT_mg/L', 'FE_TOT_mg/L'
]

def clean_numeric(val):
    if pd.isna(val):
        return np.nan
    val = str(val).strip()
    if val.startswith('<'):
        return 0  # or float(val[1:]) / 2 for midpoint
    try:
        return float(val)
    except ValueError:
        return np.nan  # catches any other unexpected strings

for col in cols_to_convert:
    df[col] = df[col].apply(clean_numeric).astype(float)

In [ ]:
df.info()

In [ ]:
df.head()

## Encode Target Variable

We map the string labels to integers so scikit-learn can work with them:

| Label | Encoded |
|---|---|
| VERDE | 0 |
| AMARILLO | 1 |
| ROJO | 2 |

In [ ]:
mapping = {'VERDE': 0, 'AMARILLO': 1, 'ROJO': 2}
df['SEMÁFORO'] = df['SEMÁFORO'].map(mapping)
df.head()

## Train / Test Split

We separate features (`X`) from the target (`y`) and split into 70% training and 30% test sets.

In [ ]:
# Entry variable
X = df.drop(columns=["SEMÁFORO"])

# Target variable
y = df["SEMÁFORO"]

X.shape, y.shape

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42
)

print("Train shapes:", X_train.shape, y_train.shape)
print("Test shapes: ", X_test.shape,  y_test.shape)

## Imputation

Some columns have missing values. We fill them with the **training set mean only** —
computing the mean on the full dataset before the split would leak information from
the test set into training, which would give overly optimistic results.

In [ ]:
df.isna().sum()

In [ ]:
# Fit on train only to prevent data leakage
imputation_features = [
    'ALC_mg/L',
    'CONDUCT_mS/cm',
    'SDT_mg/L',
    'FLUORUROS_mg/L',
    'DUR_mg/L',
    'COLI_FEC_NMP/100_mL',
    'N_NO3_mg/L',
    'AS_TOT_mg/L',
    'CD_TOT_mg/L',
    'CR_TOT_mg/L',
    'HG_TOT_mg/L',
    'PB_TOT_mg/L',
    'MN_TOT_mg/L',
    'FE_TOT_mg/L'
]

for col in imputation_features:
    train_mean = X_train[col].mean()
    X_train[col] = X_train[col].fillna(train_mean)
    X_test[col]  = X_test[col].fillna(train_mean)

print("Missing values after imputation:")
print(X_train.isna().sum().sum(), "in train —", X_test.isna().sum().sum(), "in test")

## Model: Decision Tree

We train a Decision Tree classifier with `max_depth=6` to keep the model interpretable
and avoid overfitting to the training data.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Define the model
tree = DecisionTreeClassifier(max_depth=6, random_state=42)

# Train the model
tree.fit(X_train, y_train)

## Evaluation

We evaluate the model on both the training and test sets. A small gap between the two
accuracy scores indicates the model generalizes well without overfitting.

In [ ]:
from sklearn.metrics import accuracy_score

# Training predictions
y_train_pred = tree.predict(X_train)

# Test predictions
y_pred = tree.predict(X_test)

print("Train accuracy:", accuracy_score(y_train, y_train_pred))
print("Test accuracy: ", accuracy_score(y_test,  y_pred))

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred, target_names=['VERDE', 'AMARILLO', 'ROJO']))

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_estimator(
    tree,
    X_test,
    y_test,
    display_labels=['VERDE', 'AMARILLO', 'ROJO'],
    cmap="Greens"
)
plt.title("Confusion Matrix - Decision Tree")
plt.show()